In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf

from transformers import AutoTokenizer, TFAutoModelForSequenceClassification
from scipy.special import softmax
from tqdm.notebook import tqdm
from time import sleep, time

In [2]:
df = pd.read_csv('../preprocesseddata/2020-01-01_2023-04-10.csv')

In [3]:
# Define the model and tokenizer
model_name = "cardiffnlp/twitter-roberta-base-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = TFAutoModelForSequenceClassification.from_pretrained(model_name)

# Define the labels
labels = ['Negative', 'Neutral', 'Positive']


# function to encode tweets and give out sentiment scores
def encoding_scoring(tweet):
    scores = []
    # encode tweets using the tokenizer
    encoded_tweet = tokenizer.encode_plus(
    tweet,
    add_special_tokens=True,
    max_length=128,
    padding='max_length',
    return_attention_mask=True,
    truncation=True,
    return_tensors='tf'
    )

    outputs = model(encoded_tweet)
    scores.append(tf.nn.softmax(outputs.logits, axis=-1).numpy()[0])
    return scores

All model checkpoint layers were used when initializing TFRobertaForSequenceClassification.

All the layers of TFRobertaForSequenceClassification were initialized from the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFRobertaForSequenceClassification for predictions without further training.


In [6]:
# Pass the encoded tweet through the model to get the predictions
tic = time()
sentiment_scores = []
with tqdm(total=len(df['Text'])) as pbar:
    for i, txt in enumerate(df.Text):
        sentiment_scores.append(encoding_scoring(txt))
        pbar.update(1)
# sentiment_scores = df['Text'].apply(encoding_scoring)
tac = time()
print(tac-tic)

  0%|          | 0/142651 [00:00<?, ?it/s]

62326.786724328995


In [7]:
sentiment_scores[0][0][2]

0.8219476

In [8]:
# Making negatie, neutral and positive scores columns
df['negative sentiment score'] = [score[0][0] for score in sentiment_scores]
df['neutral sentiment score'] = [score[0][1] for score in sentiment_scores]
df['positive sentiment score'] = [score[0][2] for score in sentiment_scores]

In [9]:
# Calculating compound scores
compound = []

for i in range(df.shape[0]):
    compound.append((df['negative sentiment score'][i]*-1 + df['positive sentiment score'][i])/3)
df['compound sentiment score'] = compound

In [10]:
df.head()

,Text,Datetime,Word,Make,Model,negative sentiment score,neutral sentiment score,positive sentiment score,compound sentiment score
0,@user @user @user Definitely. R1S is a much be...,2023-04-09 23:44:47+00:00,model x,Tesla,Model X,0.017797,0.160256,0.821948,0.268050
1,"Model X Long Range Demo\nKing Of Prussia, PA\n...",2023-04-09 23:23:56+00:00,model x,Tesla,Model X,0.013023,0.474754,0.512223,0.166400
2,"Model X Long Range New\nBuena Park, CA\nPearl ...",2023-04-09 23:17:51+00:00,model x,Tesla,Model X,0.010804,0.510698,0.478498,0.155898
3,"Model X Long Range New\nCarlsbad, CA\nMidnight...",2023-04-09 23:17:50+00:00,model x,Tesla,Model X,0.008921,0.457200,0.533879,0.174986
4,"Model X Long Range Demo\nIndianapolis, IN\nSol...",2023-04-09 23:17:48+00:00,model x,Tesla,Model X,0.010660,0.462582,0.526758,0.172033


In [11]:
df.to_csv('../preprocesseddata/sentiment_analysis_142651-2020-01-01_2023-04-10.csv', index=False)